# Machine Translation - Transformers

Using the tutorial of NLP of keras for machine translation with transformers, we implemented a basic model for [Spanish to English translation](https://keras.io/examples/nlp/neural_machine_translation_with_transformer/).

In [177]:
# We set the backend to TensorFlow. The code works with
# both `tensorflow` and `torch`. It does not work with JAX
# due to the behavior of `jax.numpy.tile` in a jit scope
# (used in `TransformerDecoder.get_causal_attention_mask()`:
# `tile` in JAX does not support a dynamic `reps` argument.
# You can make the code work in JAX by wrapping the
# inside of the `get_causal_attention_mask` method in
# a decorator to prevent jit compilation:
# `with jax.ensure_compile_time_eval():`.
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import pathlib
import random
import string
import re
import numpy as np

import tensorflow.data as tf_data
import tensorflow.strings as tf_strings
import tensorflow as tf

import keras
from keras import layers
from keras import ops
from keras.layers import TextVectorization

## Download the dataset

We will download the spa-eng dataset, which is a collection of Spanish-English sentence pairs.

In [178]:
text_file = keras.utils.get_file(
    fname="spa-eng.zip",
    origin="http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip",
    extract=True,
)
text_file = pathlib.Path(text_file) / "spa-eng" / "spa.txt"

## Parsing and preparing the dataset

In each line you have an English sentence and a Spanish sentence, separated by a tab. We will parse the dataset and prepare it for training. We also add two tokens: `[start]` and `[end]` to the English sentences.

In [179]:
with open(text_file, encoding='utf-8') as f:
    lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
    eng, spa = line.split("\t")
    eng = "[start] " + eng + " [end]"
    text_pairs.append((spa, eng))

for _ in range(5):
    print(random.choice(text_pairs))

('Ellos parecían estar discutiendo un tema de mucha importancia.', '[start] They seemed to be discussing a matter of great importance. [end]')
('Repite cada frase después de mí.', '[start] Repeat each sentence after me. [end]')
('Esta aspiradora hace mucho ruido.', '[start] This vacuum cleaner makes a lot of noise. [end]')
('La escurridiza serpiente se deslizó fuera de su mano.', '[start] The slippery snake slithered right out of his hand. [end]')
('¿Puedes quedarte un rato?', '[start] Can you stay for a while? [end]')


## Dividing the dataset into train/val/test sets

We will divide the dataset into train/val/test sets for training and evaluation.

In [180]:
random.shuffle(text_pairs)
#subset_pairs = text_pairs[:20000]
num_val_samples = int(0.15 * len(text_pairs))
#num_val_samples = int(0.15 * len(subset_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples
#num_train_samples = len(subset_pairs) - 2 * num_val_samples
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples : num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples :]

print(f"{len(text_pairs)} total pairs")
#print(f"{len(subset_pairs)} subset pairs")
print(f"{len(train_pairs)} training pairs")
print(f"{len(val_pairs)} validation pairs")
print(f"{len(test_pairs)} test pairs")

118964 total pairs
83276 training pairs
17844 validation pairs
17844 test pairs


## Vectorizing the text data

We have to use `TextVectorization` layers to vectorize the text data, 2 separate layers are used for each language, and specific strip characters are defined.

In [181]:
strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

vocab_size = 15000
sequence_length = 20
batch_size = 64


def custom_standardization(input_string):
    lowercase = tf_strings.lower(input_string)
    return tf_strings.regex_replace(lowercase, "[%s]" % re.escape(strip_chars), "")


eng_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)
spa_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
    standardize=custom_standardization,
)

# All the English and Spanish texts from the training set, in separate lists
train_eng_texts = [pair[1] for pair in train_pairs]
train_spa_texts = [pair[0] for pair in train_pairs]

# Create the vocabulary for both English and Spanish text vectorization layers
eng_vectorization.adapt(train_eng_texts)
spa_vectorization.adapt(train_spa_texts)

Now we have to prepare the dataset for training.

In [182]:
def format_dataset(eng, spa):
    eng = eng_vectorization(eng)
    spa = spa_vectorization(spa)
    return (
        {
            "encoder_inputs": spa,
            "decoder_inputs": eng[:, :-1],
        },
        eng[:, 1:],
    )


def make_dataset(pairs):
    spa_texts, eng_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf_data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset)
    return dataset.cache().shuffle(2048).prefetch(16)


train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

Testing our functions on a few examples, we can see that the vectorization is working as expected.

In [183]:
for inputs, targets in train_ds.take(1):
    print(f'inputs["encoder_inputs"].shape: {inputs["encoder_inputs"].shape}')
    print(f'inputs["decoder_inputs"].shape: {inputs["decoder_inputs"].shape}')
    print(f"targets.shape: {targets.shape}")

inputs["encoder_inputs"].shape: (64, 20)
inputs["decoder_inputs"].shape: (64, 20)
targets.shape: (64, 20)


## Building the transformer model

Now we will build the transformer model for machine translation.

In [184]:
import keras.ops as ops

@keras.saving.register_keras_serializable()
class CustomMultiHeadAttention(layers.Layer):
    def __init__(self, num_heads, key_dim, **kwargs):
        super().__init__(**kwargs) 
        self.num_heads = num_heads
        # key_dim is the TOTAL embedding size (e.g., 256)
        self.key_dim = key_dim 
        # Calculate size per head
        self.head_dim = key_dim // num_heads
       
        self.scale = ops.sqrt(ops.cast(self.head_dim, dtype="float32"))

        # Linear projections - output is the full embed_dim
        self.query_dense = layers.Dense(key_dim)
        self.key_dense = layers.Dense(key_dim)
        self.value_dense = layers.Dense(key_dim)
        self.output_dense = layers.Dense(key_dim)

    def split_heads(self, x, batch_size):
        # Shape: (batch, seq, heads, head_dim)
        x = ops.reshape(x, (batch_size, -1, self.num_heads, self.head_dim))
        # Shape: (batch, heads, seq, head_dim)
        return ops.transpose(x, (0, 2, 1, 3))
        
    def build(self, input_shape):
        self.query_dense.build(input_shape)
        self.key_dense.build(input_shape)
        self.value_dense.build(input_shape)
        self.output_dense.build(input_shape)
        self.built = True

    def build_from_config(self, config):
        super().build_from_config(config)

    def call(self, query, value, key, attention_mask=None):
        batch_size = ops.shape(query)[0]
        
        # 1. Projections & 2. Split
        q = self.split_heads(self.query_dense(query), batch_size)
        v = self.split_heads(self.value_dense(value), batch_size)
        k = self.split_heads(self.key_dense(key), batch_size)

        # 3. Scaled Dot-Product
        scores = ops.matmul(q, ops.transpose(k, (0, 1, 3, 2))) / self.scale

        # 4. UNIVERSAL MASK FIX
        if attention_mask is not None:
            mask = ops.cast(attention_mask, scores.dtype)
            
            # While mask has fewer dimensions than scores (4), keep adding dims at axis 1
            # If mask is (batch, seq), it becomes (batch, 1, 1, seq)
            # If mask is (batch, seq, seq), it becomes (batch, 1, seq, seq)
            while len(ops.shape(mask)) < len(ops.shape(scores)):
                mask = ops.expand_dims(mask, axis=1)
                
            mask_penalty = (1.0 - mask) * -1e9
            scores += mask_penalty
            
        # 5. Softmax & Recombine
        weights = ops.softmax(scores, axis=-1)
        attention_output = ops.matmul(weights, v)
        attention_output = ops.transpose(attention_output, (0, 2, 1, 3))
        concat_attention = ops.reshape(attention_output, (batch_size, -1, self.key_dim))

        return self.output_dense(concat_attention)

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_heads": self.num_heads,
            "key_dim": self.key_dim,
        })
        return config
    
@keras.saving.register_keras_serializable()
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim: int, dense_dim: int, num_heads: int, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = CustomMultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.supports_masking = True

    def build(self, input_shape):
        self.attention.build(input_shape)
        self.layernorm_1.build(input_shape)
        self.layernorm_2.build(input_shape)
        super().build(input_shape)

    def call(self, inputs, mask=None):
        if mask is not None:
            padding_mask = ops.cast(mask[:, None, :], dtype="int32")
        else:
            padding_mask = None

        attention_output = self.attention(
            query=inputs, value=inputs, key=inputs, attention_mask=padding_mask
        )

        # input residual and attention_output added together and normalized
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)

        # residual proj_input and FFN output added together and normalized
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
            }
        )
        return config

@keras.saving.register_keras_serializable()
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim
        )
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        length = ops.shape(inputs)[-1]
        positions = ops.arange(0, length, 1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        # The meaning of the token + The positionnal information
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        return ops.not_equal(inputs, 0)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config

@keras.saving.register_keras_serializable()
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, latent_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.latent_dim = latent_dim
        self.num_heads = num_heads
        self.supports_masking = True

        self.attention_1 = CustomMultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim,
            name="self_attention",
        )
        self.attention_2 = CustomMultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim,
            name="cross_attention",
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(latent_dim, activation="relu"),
                layers.Dense(embed_dim),
            ],
            name="dense_proj",
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        
    def build(self, input_shape):
        self.attention_1.build(input_shape[0])
        self.attention_2.build(input_shape[0])
        self.layernorm_1.build(input_shape[0])
        self.layernorm_2.build(input_shape[0])
        self.layernorm_3.build(input_shape[0])
        super().build(input_shape)

    def call(self, inputs, mask=None):
        decoder_inputs = inputs[0]
        encoder_outputs = inputs[1]

        causal_mask = self.get_causal_attention_mask(decoder_inputs)

        decoder_padding_mask = None
        encoder_padding_mask = None

        if mask is not None:
            if isinstance(mask, (list, tuple)):
                if len(mask) > 0:
                    decoder_padding_mask = mask[0]
                if len(mask) > 1:
                    encoder_padding_mask = mask[1]
            else:
                decoder_padding_mask = mask

        attn_output_1 = self.attention_1(
            query=decoder_inputs,
            value=decoder_inputs,
            key=decoder_inputs,
            attention_mask=causal_mask,
            
        )
        out_1 = self.layernorm_1(decoder_inputs + attn_output_1)

        attn_output_2 = self.attention_2(
            query=out_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=encoder_padding_mask
        )
        out_2 = self.layernorm_2(out_1 + attn_output_2)

        proj_output = self.dense_proj(out_2)
        return self.layernorm_3(out_2 + proj_output)
        
    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size = input_shape[0]
        seq_len = input_shape[1]

        i = tf.range(seq_len)[:, None]
        j = tf.range(seq_len)[None, :]
        mask = tf.cast(i >= j, dtype=tf.bool)

        mask = tf.expand_dims(mask, axis=0)
        mask = tf.broadcast_to(mask, [batch_size, seq_len, seq_len])
        return mask

    def compute_output_shape(self, input_shape):
        return input_shape[0]

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "latent_dim": self.latent_dim,
                "num_heads": self.num_heads,
            }
        )
        return config

Assembling our model together to have a full Transformer model.

In [185]:
embed_dim = 256
latent_dim = 2048
num_heads = 8

encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="encoder_inputs")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = TransformerEncoder(embed_dim, latent_dim, num_heads)(x)

decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="decoder_inputs")
encoded_seq_inputs = keras.Input(shape=(None, embed_dim), name="decoder_state_inputs")

x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
x = TransformerDecoder(embed_dim, latent_dim, num_heads)([x, encoded_seq_inputs])
x = layers.Dropout(0.5)(x)
decoder_outputs = layers.Dense(vocab_size, activation="softmax")(x)

decoder = keras.Model(
    [decoder_inputs, encoded_seq_inputs],
    decoder_outputs,
    name="decoder",
)

final_decoder_outputs = decoder([decoder_inputs, encoder_outputs])

transformer = keras.Model(
    {"encoder_inputs": encoder_inputs, "decoder_inputs": decoder_inputs},
    final_decoder_outputs,
    name="transformer",
)
transformer.summary()

Model: "transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 256) │  3,845,120 │ encoder_inputs[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_26        │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, None, 256) │  1,315,072 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal_26[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder             │ (None, None,      │  8,227,992 │ decoder_inputs[0… │
│ (Functional)        │ 15000)            │            │ transformer_enco… │
│                     │                   │            │ not_equal_26[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 13,388,184 (51.07 MB)

 Trainable params: 13,388,184 (51.07 MB)

 Non-trainable params: 0 (0.00 B)

## Training the model

In [186]:
epochs = 30  # This should be at least 30 for convergence

transformer.summary()
transformer.compile(
    "rmsprop",
    loss=keras.losses.SparseCategoricalCrossentropy(ignore_class=0),
    metrics=["accuracy"],
)
transformer.fit(train_ds, epochs=epochs, validation_data=val_ds)

Model: "transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, None, 256) │  3,845,120 │ encoder_inputs[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_26        │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, None, 256) │  1,315,072 │ positional_embed… │
│ (TransformerEncode… │                   │            │ not_equal_26[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder             │ (None, None,      │  8,227,992 │ decoder_inputs[0… │
│ (Functional)        │ 15000)            │            │ transformer_enco… │
│                     │                   │            │ not_equal_26[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 13,388,184 (51.07 MB)

 Trainable params: 13,388,184 (51.07 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'custom_multi_head_attention_13' (of type CustomMultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


1302/1302 ━━━━━━━━━━━━━━━━━━━━ 86s 53ms/step - accuracy: 0.1310 - loss: 4.5620 - val_accuracy: 0.2187 - val_loss: 2.5251
Epoch 2/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 55s 43ms/step - accuracy: 0.2236 - loss: 2.4454 - val_accuracy: 0.2370 - val_loss: 2.1623
Epoch 3/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 55s 43ms/step - accuracy: 0.2448 - loss: 1.9978 - val_accuracy: 0.2485 - val_loss: 1.9748
Epoch 4/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 55s 43ms/step - accuracy: 0.2572 - loss: 1.7782 - val_accuracy: 0.2533 - val_loss: 1.9186
Epoch 5/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 55s 43ms/step - accuracy: 0.2659 - loss: 1.6350 - val_accuracy: 0.2562 - val_loss: 1.8850
Epoch 6/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 55s 43ms/step - accuracy: 0.2708 - loss: 1.5436 - val_accuracy: 0.2572 - val_loss: 1.8908
Epoch 7/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 55s 43ms/step - accuracy: 0.2752 - loss: 1.4810 - val_accuracy: 0.2585 - val_loss: 1.8835
Epoch 8/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 55s 43ms/step - accuracy: 0.2792 - loss: 1.43

## Save the model

In [187]:
transformer.save('./best_model_spa_eng.keras')

## Decoding test sentences

Here we are going to test on real world sentences, we will use the trained model to translate Spanish sentences to English.

In [188]:
spa_vocab = eng_vectorization.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))
max_decoded_sentence_length = 20


def decode_sequence(input_sentence):
    tokenized_input_sentence = spa_vectorization([input_sentence])
    decoded_sentence = "[start]"
    for i in range(max_decoded_sentence_length):
        tokenized_target_sentence = eng_vectorization([decoded_sentence])[:, :-1]

        predictions = transformer(
            {
                "encoder_inputs": tokenized_input_sentence,
                "decoder_inputs": tokenized_target_sentence,
            }
        )

        sampled_token_index = ops.convert_to_numpy(
            ops.argmax(predictions[0, i, :])
        ).item(0)
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token

        if sampled_token == "[end]":
            break
    return decoded_sentence


test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(30):
    input_sentence = random.choice(test_eng_texts)
    translated = decode_sequence(input_sentence)
    print(f"{input_sentence} = {translated}")

Te lo envié hace dos días. = [start] i walked to you two days [end]
Yo cuidaré a este perro. = [start] ill take care of this dog [end]
Tom ni siquiera sabe de qué estás hablando. = [start] tom doesnt even know what youre talking about [end]
Tom intentó salvarme. = [start] tom tried to save [end]
Lo hice anteayer. = [start] i did it the day before before [end]
Me pregunto que pasa si aprieto este botón. = [start] i wonder what happens if i push this button [end]
Mañana hacia esta hora estaré viendo la tele. = [start] ill be watching tv about this time tomorrow [end]
¿Por qué esperar a navidad? = [start] why christmas wait for christmas [end]
Fue un momento significativo. = [start] it was a bit of tears [end]
Tom parecía tener algo en su mente. = [start] tom seemed to have something on his mind [end]
Vivo en Moscú. = [start] i live in full [end]
Me pregunto si Tom nos puede oír. = [start] i wonder if tom can hear us [end]
Creía que ya habías terminado con esto. = [start] i thought youd a

In [189]:
# 1. First, make sure you have your custom classes defined in a previous cell
# (PositionalEmbedding, TransformerEncoder, etc.)
import os

# 2. Re-create the EXACT architecture
embed_dim = 256
latent_dim = 2048
num_heads = 8
vocab_size = 15000
sequence_length = 20

encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="encoder_inputs")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = TransformerEncoder(embed_dim, latent_dim, num_heads)(x)
encoder = keras.Model(encoder_inputs, encoder_outputs)

decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="decoder_inputs")
encoded_seq_inputs = keras.Input(shape=(None, embed_dim), name="decoder_state_inputs")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
x = TransformerDecoder(embed_dim, latent_dim, num_heads)([x, encoded_seq_inputs])
x = layers.Dropout(0.5)(x)
decoder_outputs = layers.Dense(vocab_size, activation="softmax")(x)
decoder = keras.Model([decoder_inputs, encoded_seq_inputs], decoder_outputs)

decoder_outputs = decoder([decoder_inputs, encoder_outputs])
transformer = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

dummy_encoder_input = ops.zeros((1, sequence_length), dtype="int64")
dummy_decoder_input = ops.zeros((1, sequence_length), dtype="int64")

_ = transformer([dummy_encoder_input, dummy_decoder_input])

model_path = "/kaggle/working/best_model_spa_eng.keras"
transformer.load_weights(model_path)

print("Weights loaded successfully! Your transformer is ready.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'functional_37' (of type Functional) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:965: UserWarning: Layer 'custom_multi_head_attention_14' (of type CustomMultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Weights loaded successfully! Your transformer is ready.
